In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp
import torch

path = "/project/Wellcome_Discovery/datashare/lturiano/data/"
gex = sc.read_h5ad(path + "gex_filt_aligned.h5ad")  # input (Gin genes)
rna = sc.read_h5ad(path + "rna_filt_aligned.h5ad")  # target (Gout genes)
universe = pd.read_csv("genes_full_list.txt", sep=",")

In [3]:
universe

,Gene stable ID,Gene type,Gene name
0,ENSG00000210049,Mt_tRNA,MT-TF
1,ENSG00000211459,Mt_rRNA,MT-RNR1
2,ENSG00000210077,Mt_tRNA,MT-TV
3,ENSG00000210082,Mt_rRNA,MT-RNR2
4,ENSG00000209082,Mt_tRNA,MT-TL1
...,...,...,...
86364,ENSG00000168710,protein_coding,AHCYL1
86365,ENSG00000081692,protein_coding,JMJD4
86366,ENSG00000157873,protein_coding,TNFRSF14
86367,ENSG00000132676,protein_coding,DAP3


In [9]:
universe_genes = universe["Gene stable ID"].astype(str).tolist()

In [4]:
# df columns assumed: "Gene stable ID", "Gene type", "Gene name"

# assign an integer index for embeddings
universe["gene_idx"] = range(len(universe))

# lookup dicts
ensg_to_idx = dict(zip(universe["Gene stable ID"], universe["gene_idx"]))
idx_to_ensg = dict(zip(universe["gene_idx"], universe["Gene stable ID"]))

# optional: symbol lookup (gene symbols are NOT guaranteed unique)
# if you want symbol->idx, decide a rule (e.g., first match)
symbol_to_idx = (
    universe.drop_duplicates(subset=["Gene name"], keep="first")
      .set_index("Gene name")["gene_idx"]
      .to_dict()
)

universe.head()

,Gene stable ID,Gene type,Gene name,gene_idx
0,ENSG00000210049,Mt_tRNA,MT-TF,0
1,ENSG00000211459,Mt_rRNA,MT-RNR1,1
2,ENSG00000210077,Mt_tRNA,MT-TV,2
3,ENSG00000210082,Mt_rRNA,MT-RNR2,3
4,ENSG00000209082,Mt_tRNA,MT-TL1,4


In [5]:
print(gex.shape, rna.shape)
print("Same n_cells?", gex.n_obs == rna.n_obs)
ct_ok = np.array_equal(gex.obs["cell_type"].to_numpy(), rna.obs["cell_type"].to_numpy())
print("Same cell_type row-by-row:", ct_ok)

(74717, 22333) (74717, 20028)
Same n_cells? True
Same cell_type row-by-row: True


In [6]:
import numpy as np
import scipy.sparse as sp
import anndata as ad
import pandas as pd

def align_adata_to_universe_preserve(
    adata: ad.AnnData,
    universe: list[str],
    var_key: str | None = None,   # None => use adata.var_names; else use adata.var[var_key]
    dtype=np.float32,
    keep_layers: bool = False,    # if True, align each layer too (can be expensive)
):
    # --- gene ids in input (columns of X) ---
    gene_ids = (adata.var_names if var_key is None else adata.var[var_key]).astype(str).to_numpy()

    col_map = {g: i for i, g in enumerate(gene_ids)}
    idx = np.array([col_map.get(g, -1) for g in universe], dtype=np.int64)

    X = adata.X
    n = adata.n_obs
    G = len(universe)

    # --- align X ---
    if sp.issparse(X):
        X = X.tocsr().astype(dtype)
        cols = []
        for src in idx:
            if src == -1:
                cols.append(sp.csr_matrix((n, 1), dtype=dtype))
            else:
                cols.append(X[:, src])
        X_aligned = sp.hstack(cols, format="csr")
    else:
        X = np.asarray(X, dtype=dtype)
        X_aligned = np.zeros((n, G), dtype=dtype)
        keep = idx != -1
        X_aligned[:, keep] = X[:, idx[keep]]

    # --- build var metadata aligned to universe ---
    # We'll use a stable index = the gene IDs used for matching.
    # If var_key is None, adata.var_names are already gene IDs.
    var_in = adata.var.copy()
    if var_key is None:
        var_in = var_in.copy()
        var_in.index = adata.var_names.astype(str)
    else:
        # if var_key column exists, use it as the index for reindexing
        var_in = var_in.copy()
        var_in.index = adata.var[var_key].astype(str).to_numpy()

    var_aligned = var_in.reindex(universe)  # rows for missing genes become NaN
    var_aligned.index = pd.Index(universe, name=var_in.index.name)

    # --- create output AnnData and preserve metadata ---
    out = ad.AnnData(
        X=X_aligned,
        obs=adata.obs.copy(),
        var=var_aligned,
        uns=adata.uns.copy(),
        obsm=adata.obsm.copy(),
        varm=adata.varm.copy(),
        obsp=adata.obsp.copy(),
        varp=adata.varp.copy(),
    )
    out.var_names = universe

    # --- optionally align layers too ---
    if keep_layers and len(adata.layers) > 0:
        for lname, L in adata.layers.items():
            if sp.issparse(L):
                L = L.tocsr().astype(dtype)
                cols = []
                for src in idx:
                    if src == -1:
                        cols.append(sp.csr_matrix((n, 1), dtype=dtype))
                    else:
                        cols.append(L[:, src])
                out.layers[lname] = sp.hstack(cols, format="csr")
            else:
                L = np.asarray(L, dtype=dtype)
                L_al = np.zeros((n, G), dtype=dtype)
                keep = idx != -1
                L_al[:, keep] = L[:, idx[keep]]
                out.layers[lname] = L_al

    return out, idx

In [10]:
gex_aligned, idx = align_adata_to_universe_preserve(gex, universe_genes, var_key=None)
print(gex_aligned.shape)

(74717, 86369)


In [11]:
rna_aligned, idx = align_adata_to_universe_preserve(rna, universe_genes, var_key=None)
print(rna_aligned.shape)

(74717, 86369)


In [17]:
gex_aligned.obs

,cell_type,organ
AAACAGCCAATATACC-1-1045-01_1_1,T cell,lymph
AAACAGCCAATTTGGT-1-1118-01_1_1,T cell,lymph
AAACAGCCACCTGTAA-1-1120-01_1_1,T cell,lymph
AAACAGCCAGGAATCG-1-1118-01_1_1,T cell,lymph
AAACAGCCAGGACACA-1-1045-01_1_1,T cell,lymph
...,...,...
HCAHeartST13189997_HCAHeartST13188802_TCCCGGACAGAAATGC-1,neural cell,heart
HCAHeartST13189997_HCAHeartST13188802_TCTAGCCTCAAGCGCC-1,neural cell,heart
HCAHeartST13189997_HCAHeartST13188802_TGAGGCACACGGTTTA-1,neural cell,heart
HCAHeartST13189997_HCAHeartST13188802_TGTCATAAGCTAAAGG-1,neural cell,heart


In [18]:
rna_aligned.obs

,cell_type,organ
AAACCTGAGGCACATG-1-Human_colon_16S8117828,T cell,lymph
AAACCTGAGTAGCGGT-1-Human_colon_16S8117828,T cell,lymph
AAACCTGGTAGAGCTG-1-4861STDY7135913,T cell,lymph
AAACGGGAGTATGACA-1-4861STDY7135913,T cell,lymph
AAACGGGTCTCCAACC_Donor2_lymph node_act,T cell,lymph
...,...,...
Sample1162-EO2_CCACGTTAGATGGGCT-1,neural cell,kidney
Sample1162-EO2_GTGTTAGCACCAGTAT-1,neural cell,kidney
Sample1162-EO2_TATATCCGTCTGTCCT-1,neural cell,kidney
Sample1162-EO2_TATTGCTTCGTTACCC-1,neural cell,kidney


In [20]:
gex_aligned.write_h5ad(path + "gex_filt_aligned.h5ad")
rna_aligned.write_h5ad(path + "rna_filt_aligned.h5ad")